In [9]:
import numpy as np

class SoilSimulator:
    def __init__(self, initial_moisture=700):
        self.curr_moisture = float(initial_moisture)
        self.drying_rate = 0.5        # How much it dries per "tick"
        self.pump_efficiency = 20.0   # Base water impact
        self.last_decrease = 0.0

    def tick(self, pump_is_on=False):
        """Simulates one second of time passing."""

        # 1. Natural Drying (Reading goes UP)
        # Adds a little bit of randomness to how fast it dries
        rng = np.random.default_rng()
        drying_jitter = rng.normal(loc=0, scale=0.1)
        self.curr_moisture += (self.drying_rate + drying_jitter)

        # 2. Watering (Reading goes DOWN)
        if pump_is_on:
            # The pump efficiency fluctuates slightly (e.g., pressure changes)
            flow_jitter = np.random.normal(loc=0, scale=1.5)
            self.last_decrease = self.pump_efficiency + flow_jitter
            self.curr_moisture -= self.last_decrease
        else:
          self.last_decrease = 0.0 # No pumping = 0 decrease

        # Keep moisture in a realistic 10-bit range (0-1023)
        # Moisture cannot go below 200 in realistic situations
        self.curr_moisture = np.clip(self.curr_moisture, 200, 1023)

    def get_sensor_reading(self):
        """Simulates the noisy electrical signal from the sensor."""
        # Electrical noise: the sensor flickers by +/- 2 units constantly
        electrical_noise = np.random.normal(loc=0, scale=2.0)
        return int(self.curr_moisture + electrical_noise), int(self.last_decrease)

In [17]:
import pandas as pd

# --- EXECUTION ---
sim = SoilSimulator(initial_moisture=700)
data = []

SIMULATION_TIME = 10

# Simulates for a minute
for s in range(SIMULATION_TIME + 1):
  pumping = True if s >= 5 else False

  if s > 0:
    sim.tick(pump_is_on=pumping)

  moisture, dec = sim.get_sensor_reading()
  time_label = "Dry" if s == 0 else f"{s}s"
  data.append((time_label, moisture, dec))

df = pd.DataFrame(data, columns=["Total pump time", "Soil Moisture", "Decrease"])

print(df)

# Find average
avg = df.loc[df["Decrease"] > 0, "Decrease"].mean()
print(f"The average decrease is {avg:.2f}")

   Total pump time  Soil Moisture  Decrease
0              Dry            700         0
1               1s            700         0
2               2s            704         0
3               3s            699         0
4               4s            703         0
5               5s            681        19
6               6s            657        25
7               7s            636        23
8               8s            612        22
9               9s            592        19
10             10s            578        18
The average decrease is 21.00


In [18]:
def get_pump_duration(current_moisture, target_moisture, calibrated_avg):
  if current_moisture <= target_moisture:
    return 0.0  # Already wet enough!

  duration = (current_moisture - target_moisture) / calibrated_avg
  return round(duration, 2)

# example of using this data
target = 400
current_reading, _ = sim.get_sensor_reading()

seconds_to_run = get_pump_duration(current_reading, target, avg)

print(f"Server Decision: Soil is at {current_reading}. "
      f"To reach {target}, running pump for {seconds_to_run}s.")

Server Decision: Soil is at 578. To reach 400, running pump for 8.48s.
